# Ablações de variáveis em val_ref

Treina cada conjunto de colunas da tabela de ablações e mede o efeito sobre o conjunto P.
Lê as partições `train_ref` e `val_ref` do artefato supervisionado final.
Grava duas tabelas em `outputs/final/`: resultados por linha e ROC-AUC por ano.

In [1]:
from pathlib import Path
import sys


def encontrar_raiz_projeto(start: Path | None = None) -> Path:
    atual = (start or Path.cwd()).resolve()
    for candidato in (atual, *atual.parents):
        if (candidato / "src").is_dir() and (candidato / "pyproject.toml").is_file():
            return candidato
    raise RuntimeError(f"Raiz do projeto não encontrada a partir de: {atual}")


ROOT_DIR = encontrar_raiz_projeto()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.utils import config as C

In [2]:
from src.data.supervised_artifact import load_supervised_artifact

df, manifest = load_supervised_artifact(ROOT_DIR, expected_kind="final", partitions=("train_ref", "val_ref"))
print(df.shape, df[manifest["partition_column"]].value_counts().to_dict())

(596210, 46) {'train_ref': 556462, 'val_tune': 39748}


In [3]:
import time

import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline

from src.evaluation.metrics import auc, average_precision, bootstrap_grupo, brier
from src.features.engineering import recodificar_comorbidades_por_fator_risco
from src.models.factory import build_finalista
from src.models.pipeline import build_preprocessor
from src.utils import config as C  # noqa: N812

TRES_SINTOMAS = ["perd_olft", "perd_pala", "fadiga"]
COMORBIDADES = ["cardiopati", "hematologi", "sind_down", "hepatica", "asma", "diabetes", "neurologic",
                "pneumopati", "imunodepre", "renal", "obesidade", "out_morbi", "puerpera"]


def pipeline_ablacao(familia, numericas, categoricas):
    """Pipeline com hiperparâmetros padrão para um conjunto de colunas escolhido."""
    if familia == "logistic":
        return build_finalista("logistic", preprocessor=build_preprocessor(numericas, categoricas))
    mascara = [False] * len(numericas) + [True] * len(categoricas)
    classificador = HistGradientBoostingClassifier(
        learning_rate=0.08, max_iter=150, max_leaf_nodes=31, min_samples_leaf=100, l2_regularization=0.10,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, class_weight="balanced",
        categorical_features=mascara, random_state=C.SEED,
    )
    return Pipeline([("pre", build_preprocessor(numericas, categoricas, saida="ordinal")), ("clf", classificador)])


def executar_ablacoes(df, manifest):
    """Treina em train_ref e mede em val_ref cada linha da tabela de ablações."""
    col_part, alvo = manifest["partition_column"], manifest["target_column"]
    num, cat = list(manifest["numeric_columns"]), list(manifest["categorical_columns"])
    treino, valid = df[col_part] == "train_ref", df[col_part] == "val_ref"
    y_tr, y_va = df.loc[treino, alvo].to_numpy(), df.loc[valid, alvo].to_numpy()
    prevalencia = float(y_va.mean())

    base = df[COMORBIDADES + ["fator_risc"]].astype("object")  # astype: evita erro de categoria nova ("2")
    recod = recodificar_comorbidades_por_fator_risco(base)
    df_recod = df.drop(columns=COMORBIDADES).join(recod[COMORBIDADES])

    sem3 = lambda cols: [c for c in cols if c not in TRES_SINTOMAS]  # noqa: E731
    linhas = {
        "P": (df, num, cat, ["logistic", "boosting"]),
        "P_menos_3_sintomas": (df, sem3(num), sem3(cat), ["logistic", "boosting"]),
        "P_comorbidades_recodificadas": (df_recod, num, cat, ["logistic", "boosting"]),
        "P_mais_gravidade": (df, num, cat + ["saturacao", "desc_resp"], ["logistic"]),
        "P_mais_contexto": (df, num, cat + ["cs_escol_n", "vacina"], ["logistic"]),
        "P_mais_regiao": (df, num, cat + ["regiao_residencia"], ["logistic"]),
    }
    resultados, probabilidades = [], {}
    for linha, (quadro, n_cols, c_cols, familias) in linhas.items():
        for familia in familias:
            t0 = time.perf_counter()
            pipe = pipeline_ablacao(familia, n_cols, c_cols)
            pipe.fit(quadro.loc[treino, n_cols + c_cols], y_tr)
            prob = pipe.predict_proba(quadro.loc[valid, n_cols + c_cols])[:, 1].astype("float32")
            probabilidades[(linha, familia)] = prob
            pr = average_precision(y_va, prob)
            resultados.append({"linha": linha, "familia": familia, "n_colunas": len(n_cols + c_cols),
                               "pr_auc": pr, "pr_auc_sobre_prevalencia": pr / prevalencia,
                               "roc_auc": auc(y_va, prob), "brier": brier(y_va, prob),
                               "duracao_s": round(time.perf_counter() - t0, 1)})
            del pipe
    tabela = pd.DataFrame(resultados)
    ref = tabela[tabela["linha"] == "P"].set_index("familia")
    tabela["dif_pr_auc_vs_P"] = tabela.apply(lambda r: r["pr_auc"] - ref.loc[r["familia"], "pr_auc"], axis=1)
    tabela["dif_roc_auc_vs_P"] = tabela.apply(lambda r: r["roc_auc"] - ref.loc[r["familia"], "roc_auc"], axis=1)

    anos = df.loc[valid, "ano_internacao"].to_numpy()
    grupos = df.loc[valid, "group_id"].to_numpy()
    por_ano = []
    for ano in sorted(set(anos)):
        m = anos == ano
        linha_ano = {"ano": int(ano), "n": int(m.sum()), "obitos": int(y_va[m].sum())}
        for familia in ("logistic", "boosting"):
            p_p, p_3 = probabilidades[("P", familia)][m], probabilidades[("P_menos_3_sintomas", familia)][m]
            linha_ano[f"roc_auc_P_{familia}"] = auc(y_va[m], p_p)
            linha_ano[f"roc_auc_menos3_{familia}"] = auc(y_va[m], p_3)
            ic = bootstrap_grupo(y_va[m], p_3, grupos[m], metrica="auc", prob_b=p_p, n=300)
            linha_ano[f"dif_menos3_menos_P_{familia}"] = ic["estimativa_diferenca"]
            linha_ano[f"ic_inf_{familia}"], linha_ano[f"ic_sup_{familia}"] = ic["ic_inferior"], ic["ic_superior"]
        por_ano.append(linha_ano)
    return tabela, pd.DataFrame(por_ano)

In [4]:
from IPython.display import display

tabela, por_ano = executar_ablacoes(df, manifest)
display(tabela.round(4))
display(por_ano.round(4))

,linha,familia,n_colunas,pr_auc,pr_auc_sobre_prevalencia,roc_auc,brier,duracao_s,dif_pr_auc_vs_P,dif_roc_auc_vs_P
0,P,logistic,31,0.3031,3.5907,0.8482,0.1655,13.2,0.0000,0.0000
1,P,boosting,31,0.3348,3.9670,0.8596,0.1640,25.6,0.0000,0.0000
2,P_menos_3_sintomas,logistic,28,0.3029,3.5884,0.8478,0.1655,14.6,-0.0002,-0.0003
3,P_menos_3_sintomas,boosting,28,0.3345,3.9626,0.8592,0.1642,21.0,-0.0004,-0.0004
4,P_comorbidades_recodificadas,logistic,31,0.3031,3.5907,0.8482,0.1655,12.5,0.0000,0.0000
5,P_comorbidades_recodificadas,boosting,31,0.3348,3.9670,0.8596,0.1640,26.6,0.0000,0.0000
6,P_mais_gravidade,logistic,33,0.3226,3.8223,0.8569,0.1607,15.1,0.0195,0.0088
7,P_mais_contexto,logistic,33,0.3064,3.6295,0.8499,0.1648,13.0,0.0033,0.0018
8,P_mais_regiao,logistic,32,0.3063,3.6293,0.8493,0.1650,14.3,0.0033,0.0012


,ano,n,obitos,roc_auc_P_logistic,roc_auc_menos3_logistic,dif_menos3_menos_P_logistic,ic_inf_logistic,ic_sup_logistic,roc_auc_P_boosting,roc_auc_menos3_boosting,dif_menos3_menos_P_boosting,ic_inf_boosting,ic_sup_boosting
0,2019,2169,260,0.7926,0.7927,0.0001,-0.0007,0.0008,0.8032,0.8037,0.0005,-0.0028,0.0041
1,2023,11899,1124,0.8542,0.8544,0.0002,-0.0004,0.0009,0.8669,0.8664,-0.0005,-0.0014,0.0005
2,2024,11338,932,0.8479,0.8476,-0.0003,-0.0010,0.0004,0.8613,0.8612,-0.0001,-0.0010,0.0011
3,2025,14342,1039,0.8493,0.8485,-0.0008,-0.0015,-0.0001,0.8582,0.8577,-0.0005,-0.0015,0.0005


In [5]:
SAIDA_DIR = ROOT_DIR / "outputs" / "final"
SAIDA_DIR.mkdir(parents=True, exist_ok=True)

tabela.to_csv(SAIDA_DIR / "ablacoes_val_tune.csv", index=False)
por_ano.to_csv(SAIDA_DIR / "ablacoes_roc_por_ano.csv", index=False)

## Decisão proposta

**Três sintomas (perd_olft, perd_pala, fadiga):** manter/retirar, porque …

**Comorbidades recodificadas:** manter/recodificar, porque …